1. Traer la receta común (constantes y funciones de home).

In [9]:
%run 1_home.ipynb

2. Cargar el parquet de 2019 y armar lag y delta 1 y 2. Pasar de 152 columnas a 760 features.

In [10]:
df = cargar_datos()
datos, features = agregar_lags_deltas(df)
print(datos.shape, len(features))

(1563565, 763) 760


3. Recuperar el estudio del bucket si la VM es nueva y el sqlite local no existe.
Tres situaciones:
- El sqlite local existe (tu caso ahora): no hace nada. La 4 ve 50 trials y no optimiza.
- No está local pero sí en el bucket (VM nueva): lo trae. La 4 ve los 50 trials y tampoco optimiza.
- No está en ningún lado (todo de cero): no hace nada. La 4 crea el estudio y corre los 50 trials, dos horas.

In [11]:
import os, shutil

ESTUDIO_LOCAL = "/home/ds/datasets/optuna_baseline.db"
ESTUDIO_BUCKET = "/home/ds/buckets/b1/optuna_baseline.db"

if not os.path.exists(ESTUDIO_LOCAL) and os.path.exists(ESTUDIO_BUCKET):
    shutil.copy(ESTUDIO_BUCKET, ESTUDIO_LOCAL)
    print("estudio restaurado del bucket")

4. Optimización bayesiana. Train 201903–201906, valida en 201908 por AUC. Si el estudio ya tiene 50 trials no hace nada; si está vacío tarda...

In [12]:
estudio = optimizar(datos, features, "baseline_colaborativos",
                    storage=f"sqlite:///{ESTUDIO_LOCAL}")

[I 2026-11-17 09:39:37,122] Using an existing study with `study_name='baseline_colaborativos'` instead of creating a new one.


[baseline_colaborativos] trials completos: 54, faltan: 0
[baseline_colaborativos] mejor AUC 0.93286
[baseline_colaborativos] {'num_leaves': 218, 'learning_rate': 0.01817804191127569, 'min_data_in_leaf': 1540, 'feature_fraction': 0.8868602506438694, 'bagging_fraction': 0.6572947358317337}


5. Respaldar el estudio en el bucket, que es lo único que sobrevive si se borra la VM.

In [13]:
shutil.copy(ESTUDIO_LOCAL, ESTUDIO_BUCKET)
print("estudio respaldado en el bucket")

estudio respaldado en el bucket


6. El modelo final con las 5 semillas. Reentrena incluyendo 201907 y 201908, predice 201910 y mide ganancia. Unos 17 minutos.

In [14]:
res, curvas, probas = correr_semillas(datos, features, estudio)
res.to_csv("/home/ds/buckets/b1/baseline_5semillas.csv", index=False)
res

  semilla 100129: $296,312,500 con 4,025 envios (257s)
  semilla 100183: $285,037,500 con 3,075 envios (208s)
  semilla 100151: $287,980,000 con 3,768 envios (209s)
  semilla 500041: $287,017,500 con 3,763 envios (235s)
  semilla 900007: $293,012,500 con 4,945 envios (211s)
media $289,872,000  desvio $4,648,737


,semilla,ganancia,envios
0,100129,296312500.0,4025
1,100183,285037500.0,3075
2,100151,287980000.0,3768
3,500041,287017500.0,3763
4,900007,293012500.0,4945


7. Curva de ganancia de las 5 semillas: dónde está la meseta y cuántos envíos conviene hacer.

In [15]:
grafico_ganancia(curvas, "Curva de ganancia en 201910 — baseline").show()

8. ROC en 201910, con BAJA+2 como clase positiva.

In [16]:
grafico_roc(datos, probas).show()

In [17]:
# --- 20 semillas: 15 nuevas que se suman a las 5 que ya estan ---
NOMBRE  = "baseline"   # "baseline" | "ipc"           | "grupos"
ESTUDIO = "baseline"   # "baseline" | "deflactado_ipc" | "deflactado_grupos"

SEMILLAS_EXTRA = [100193, 100207, 100213, 100237, 100267,
                  100271, 100279, 100291, 100297, 100313,
                  500069, 500083, 500107, 500111, 900017]

import optuna, shutil, pandas as pd
shutil.copy(f"/home/ds/buckets/b1/optuna_{NOMBRE}.db",
            f"/home/ds/datasets/optuna_{NOMBRE}.db")
est = optuna.load_study(study_name=ESTUDIO,
                        storage=f"sqlite:////home/ds/datasets/optuna_{NOMBRE}.db")

extra, _, _ = correr_semillas(datos, features, est, semillas=SEMILLAS_EXTRA)
viejas = pd.read_csv(f"/home/ds/buckets/b1/{NOMBRE}_5semillas.csv")
res20  = pd.concat([viejas, extra], ignore_index=True)
res20.to_csv(f"/home/ds/buckets/b1/{NOMBRE}_20semillas.csv", index=False)

print(f"{NOMBRE}: media ${res20.ganancia.mean():,.0f}  "
      f"desvio ${res20.ganancia.std():,.0f}  n={len(res20)}")

KeyError: 'Record does not exist.'